In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [5]:
%load_ext autoreload
%autoreload 2


from przygotowanie_danych_zuzycia_energii import przygotuj_dane_zuzycia_energii

models_path = Path.cwd().parents[0]
sys.path.append(str(models_path / "Predykcja przepompowanych ścieków"))

from przygotowanie_danych_przepompowania import przygotuj_dane_przepompowania_z_opadami



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
X_energia, Y_energia = przygotuj_dane_zuzycia_energii()
X_przeplyw, Y_przeplyw = przygotuj_dane_przepompowania_z_opadami()


In [9]:
X_energia

,hour,dayofweek,month,weekend,opad_1h,opad_3h,opad_6h,opad_12h,opad_24h,opad_48h,...,opad_168h,opad_max3h,h_od_deszczu,energia_lag_1h,energia_lag_3h,energia_lag_6h,energia_lag_24h,energia_lag_168h,energia_mean_6h,energia_mean_24h
Data_czas,,,,,,,,,,,,,,,,,,,,,
2024-10-13 01:00:00,1,6,10,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-13 02:00:00,2,6,10,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-13 03:00:00,3,6,10,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-13 04:00:00,4,6,10,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN
2024-10-13 05:00:00,5,6,10,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2.0,2.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-12 20:00:00,20,0,10,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,3.0,2.0,5.0,5.0,3.166667,2.791667
2026-10-12 21:00:00,21,0,10,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,5.0,4.0,4.0,3.0,4.0,3.666667,2.791667
2026-10-12 22:00:00,22,0,10,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,3.0,3.0,4.0,3.0,3.500000,2.791667


In [11]:
Y_energia

Data_czas
2024-10-13 01:00:00    3.0
2024-10-13 02:00:00    2.0
2024-10-13 03:00:00    3.0
2024-10-13 04:00:00    2.0
2024-10-13 05:00:00    3.0
                      ... 
2026-10-12 20:00:00    5.0
2026-10-12 21:00:00    3.0
2026-10-12 22:00:00    4.0
2026-10-12 23:00:00    3.0
2026-10-13 00:00:00    0.0
Name: zuzycie_energii, Length: 17516, dtype: float64

In [12]:
X_przeplyw

,hour,dayofweek,opad_0h,opad_2h,opad_3h,opad_6h,opad_12h,opad_24h,opad_48h,opad_max3h,h_od_deszczu
Data_czas,,,,,,,,,,,
2024-10-13 01:00:00,1,6,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-13 02:00:00,2,6,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,NaN
2024-10-13 03:00:00,3,6,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,NaN
2024-10-13 04:00:00,4,6,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,NaN
2024-10-13 05:00:00,5,6,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-10-12 20:00:00,20,0,0.0,0.0,0.0,0.0,0.2,0.2,0.5,0.0,50.0
2026-10-12 21:00:00,21,0,0.0,0.0,0.0,0.0,0.2,0.2,0.5,0.0,51.0
2026-10-12 22:00:00,22,0,0.0,0.0,0.0,0.0,0.1,0.2,0.5,0.0,52.0


In [13]:
Y_przeplyw

Data_czas
2024-10-13 01:00:00    10.0
2024-10-13 02:00:00    20.0
2024-10-13 03:00:00    10.0
2024-10-13 04:00:00    20.0
2024-10-13 05:00:00    10.0
                       ... 
2026-10-12 20:00:00    30.0
2026-10-12 21:00:00    20.0
2026-10-12 22:00:00    20.0
2026-10-12 23:00:00    20.0
2026-10-13 00:00:00     0.0
Name: przepompowano, Length: 17518, dtype: float64

In [17]:
features = [
        "hour", "dayofweek", "month", "weekend",
        "opad_1h", "opad_3h", "opad_6h", "opad_12h",
        "opad_24h", "opad_48h", "opad_72h", "opad_168h",
        "opad_max3h", "h_od_deszczu"
    ]
X_energia = X_energia[features]
X_energia

,hour,dayofweek,month,weekend,opad_1h,opad_3h,opad_6h,opad_12h,opad_24h,opad_48h,opad_72h,opad_168h,opad_max3h,h_od_deszczu
Data_czas,,,,,,,,,,,,,,
2024-10-13 01:00:00,1,6,10,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-13 02:00:00,2,6,10,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-13 03:00:00,3,6,10,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-13 04:00:00,4,6,10,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-13 05:00:00,5,6,10,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-12 20:00:00,20,0,10,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-12 21:00:00,21,0,10,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-12 22:00:00,22,0,10,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
